# Step 4. Plot and package CTD profiles

Makes publication-style vertical profile plots, per cruise:

* **CTD variables** from the SBE 1 dbar down-cast bins (`L3/CTD/CTD_1m_down/*.cnv`): temperature, salinity, oxygen,
  fluorescence, beam attenuation, density.
* **Nitrate and calibrated chlorophyll** from step08's L3 grid (`L3/gridded/<cast>_L3_1m_down.csv`), which holds the
  production nitrate (step03_nitrate_qc) and step07's chlorophyll on the same SBE bins, joined bin by bin on pressure.
  Until Nov 2026 step04 plotted the cross-check nitrate (`L2/SUNA_calibrated`) instead, matched by rounding pressure
  onto depth; that product is now only a fallback when a cast has no grid, and the run says so.

Outputs: per-cast figures and per-variable overlays in `_figures/profiles`, and a lab-friendly CSV per cast (readable
column names) in the `profile_tables` product (`L3/CTD/profiles_readable`), so profiles can be shared without the
Sea-Bird codes. Run step08 first.


## 1. Imports and settings

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # file output; no interactive backend needed
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


In [ ]:
# ===========================================================================
# Paths (match step01/02/03 v2 layout)
# ===========================================================================
import sys
from pathlib import Path
try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root, product_dir

CTD_ROOT     = project_root()
CRUISES_ROOT = CTD_ROOT / "cruises"

CRUISE_ID  = "P45_06"
CRUISE_DIR = CRUISES_ROOT / CRUISE_ID

L2_1M_DOWN   = product_dir(CRUISE_ID, "ctd_grid_1m_down")   # CTD downcast profiles (.cnv)
L2_SUNA_ROOT = product_dir(CRUISE_ID, "suna_l2")                  # merged CTD+SUNA (raw nitrate)
L3_SUNA_ROOT = product_dir(CRUISE_ID, "suna_calibrated")                  # cross-check nitrate: fallback only
L3_GRID_ROOT = product_dir(CRUISE_ID, "grid_1m_down")             # step08: production nitrate + calibrated chl
PLOT_ROOT    = product_dir(CRUISE_ID, "profile_plots")
TABLE_ROOT   = product_dir(CRUISE_ID, "profile_tables", create=True)   # the readable tables: data, not figures
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

CTD_VARIABLES = [
    (["tv290C", "t090C", "t090Cm", "potemp090C"], "Temperature", "\u00b0C"),
    (["sal00", "sal11"],                          "Salinity",    "PSU"),
    (["sbox0Mm/Kg", "sbeox0Mm/Kg", "sbeox0Mm/L"], "Oxygen",      "\u00b5mol/kg"),
    (["flECO-AFL", "flECO-AFL1"],                 "Fluorescence","mg/m\u00b3"),
    (["chl_cal"],                                 "Chlorophyll (calibrated)", "mg/m\u00b3"),
    (["CStarAt0"],                                "Beam attenuation", "1/m"),
    (["sigma-\u00e900", "sigma-t00", "density00"],"Density (sigma-t)", "kg/m\u00b3"),
]
NITRATE_VARIABLE = (["nitrate_corr", "suna_nitrate"], "Nitrate (corrected)", "\u00b5M")

DEPTH_CANDIDATES = ["depSM", "depSW", "prdM", "prDM", "prM", "pressure"]

FIG_DPI = 200
LINE_W  = 1.4
GRID_A  = 0.3
SAVE_FORMATS = ["png"]

LAB_RENAME = {
    "depSM": "Depth_m", "prdM": "Pressure_db", "tv290C": "Temperature_C",
    "sal00": "Salinity_PSU", "sbox0Mm/Kg": "Oxygen_umol_kg",
    "sbeox0Mm/L": "Oxygen_umol_L", "flECO-AFL": "Fluorescence_mg_m3",
    "CStarAt0": "BeamAttenuation_1_m", "par/sat/log": "PAR",
    "potemp090C": "PotentialTemp_C", "sigma-\u00e900": "SigmaT_kg_m3",
    "utc_time": "UTC_time", "suna_nitrate": "Nitrate_raw_uM", "nitrate_corr": "Nitrate_corrected_uM",
    "nitrate_hyst_flag": "Nitrate_downup_over_2uM", "chl_cal": "Chl_calibrated_mg_m3",
}

print("Cruise      :", CRUISE_ID)
print("CTD 1m down :", L2_1M_DOWN, "(exists:", L2_1M_DOWN.exists(), ")")
print("Merged SUNA :", L2_SUNA_ROOT, "(exists:", L2_SUNA_ROOT.exists(), ")")
print("Plots out   :", PLOT_ROOT)

## 2. Readers

A minimal `.cnv` parser (same header logic as step02/03) for the CTD downcast profiles, and a helper
to pick the depth column and available variables from any profile dataframe.

In [ ]:
def parse_cnv(path: Path) -> pd.DataFrame:
    """Parse a Sea-Bird .cnv into a numeric DataFrame using the header column names."""
    text = path.read_text(encoding="latin-1", errors="replace").splitlines()
    names: Dict[int, str] = {}
    data_start = None
    name_re = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.IGNORECASE)
    for idx, line in enumerate(text):
        s = line.strip()
        if s.startswith("#") or s.startswith("*"):
            m = name_re.search(line)
            if m:
                names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"):
                data_start = idx + 1
                break
    if data_start is None:
        raise ValueError(f"No *END* in {path}")
    cols = [names[k] for k in sorted(names)]
    rows = []
    for line in text[data_start:]:
        s = line.strip()
        if not s:
            continue
        parts = s.split()
        if len(parts) >= len(cols):
            rows.append(parts[:len(cols)])
    return pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce")


def pick_depth(df: pd.DataFrame) -> Optional[str]:
    for c in DEPTH_CANDIDATES:
        if c in df.columns:
            return c
    # loose fallback
    return next((c for c in df.columns if re.search(r"dep|pr[dD]?M|pressure", c, re.I)), None)


def pick_var(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


## 3. Plot helpers

`plot_single` draws one variable vs depth (depth increasing downward). `plot_overlay` overlays one
variable across all casts. Both save to `PLOT_ROOT` in the configured formats.

In [ ]:
def _finish_axis(ax, xlabel, ylabel="Depth (m)"):
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=GRID_A)
    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_top()


def _save(fig, base: Path) -> List[Path]:
    out = []
    for fmt in SAVE_FORMATS:
        p = base.with_suffix(f".{fmt}")
        fig.savefig(p, dpi=FIG_DPI, bbox_inches="tight")
        out.append(p)
    plt.close(fig)
    return out


def plot_single(df: pd.DataFrame, xcol: str, depth_col: str, label: str, unit: str,
                cast_id: str) -> List[Path]:
    d = df[[xcol, depth_col]].apply(pd.to_numeric, errors="coerce").dropna().sort_values(depth_col)
    if d.empty:
        return []
    fig, ax = plt.subplots(figsize=(3.2, 5.0))
    ax.plot(d[xcol], d[depth_col], "-", lw=LINE_W)
    _finish_axis(ax, f"{label} ({unit})")
    ax.set_title(cast_id, fontsize=9)
    slug = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_").lower()
    return _save(fig, PLOT_ROOT / f"{cast_id}__{slug}")


def plot_overlay(frames: Dict[str, pd.DataFrame], candidates: List[str],
                 label: str, unit: str) -> List[Path]:
    fig, ax = plt.subplots(figsize=(4.0, 6.0))
    n = 0
    for cast_id, df in sorted(frames.items()):
        xcol = pick_var(df, candidates)
        depth_col = pick_depth(df)
        if xcol is None or depth_col is None:
            continue
        d = df[[xcol, depth_col]].apply(pd.to_numeric, errors="coerce").dropna().sort_values(depth_col)
        if d.empty:
            continue
        ax.plot(d[xcol], d[depth_col], "-", lw=1.0, label=cast_id)
        n += 1
    if n == 0:
        plt.close(fig)
        return []
    _finish_axis(ax, f"{label} ({unit})")
    ax.set_title(f"{CRUISE_ID} \u2014 {label} (all casts)", fontsize=10)
    ax.legend(fontsize=6, loc="best")
    slug = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_").lower()
    return _save(fig, PLOT_ROOT / f"_overlay__{slug}")


## 4. Load all casts

Reads every 1 m down-cast CTD profile and attaches, from step08's grid for the same cast, the production nitrate
(`nitrate_corr`), its down/up flag (`nitrate_hyst_flag`) and the calibrated chlorophyll (`chl_cal`), joined on the
bin pressure. Without a grid it falls back to the cross-check nitrate, then the raw merged nitrate, and says which.


In [ ]:
PRESSURE_CANDIDATES = ["prdM", "prDM", "prSM", "prM"]
NITRATE_SOURCE: Dict[str, str] = {}


def _pressure_col(df: pd.DataFrame) -> Optional[str]:
    return pick_var(df, PRESSURE_CANDIDATES)


def attach_nitrate(ctd_df: pd.DataFrame, cast_id: str, depth_col: str) -> pd.DataFrame:
    """Attach nitrate (and calibrated chlorophyll) to the CTD bins.
    (1) step08's grid for this cast: same SBE bins, joined on the bin pressure (`ctd_pressure_db`, the SBE file's own
        value): the production nitrate, its down/up flag, and step07's chlorophyll.
    (2) fallback, no grid: the cross-check nitrate (L2/SUNA_calibrated), averaged per 1 dbar of pressure.
    (3) fallback: the raw merged SUNA nitrate, the same way. NITRATE_SOURCE records which, per cast."""
    out = ctd_df.copy()
    pcol = _pressure_col(out)
    grid = L3_GRID_ROOT / f"{cast_id}_L3_1m_down.csv"
    if grid.exists() and pcol is not None:
        g = pd.read_csv(grid)
        key = "ctd_pressure_db" if "ctd_pressure_db" in g.columns else "pressure_db"
        g = g.assign(_k=pd.to_numeric(g[key], errors="coerce").round(3)).dropna(subset=["_k"]).drop_duplicates("_k")
        g = g.set_index("_k")
        k = pd.to_numeric(out[pcol], errors="coerce").round(3)
        for src, dst in (("nitrate_uM", "nitrate_corr"), ("nitrate_hyst_flag", "nitrate_hyst_flag"),
                         ("chl_mg_m3", "chl_cal")):
            if src in g.columns:
                out[dst] = k.map(g[src]).to_numpy()
        unmatched = int(k.notna().sum() - k.isin(g.index).sum())
        NITRATE_SOURCE[cast_id] = "gridded (step08)" + (f", {unmatched} bins not in the grid" if unmatched else "")
        if "nitrate_corr" not in out or out["nitrate_corr"].isna().all():
            NITRATE_SOURCE[cast_id] = "gridded (step08), no nitrate"
        return out
    def _per_dbar(d, value):
        pc = "pressure_db" if "pressure_db" in d.columns else _pressure_col(d)
        if pc is None or value not in d.columns or pcol is None:
            return None
        dd = d[[pc, value]].apply(pd.to_numeric, errors="coerce").dropna()
        if dd.empty:
            return None
        dd["_bin"] = dd[pc].round().astype(int)
        return pd.to_numeric(out[pcol], errors="coerce").round().astype("Int64").map(dd.groupby("_bin")[value].mean())
    l3 = L3_SUNA_ROOT / f"{CRUISE_ID}__suna_L3_corrected.csv"
    if l3.exists():
        d = pd.read_csv(l3)
        d = d[d["cast"] == cast_id] if "cast" in d.columns else d
        v = _per_dbar(d, "nitrate_corr_uM")
        if v is not None and v.notna().any():
            out["nitrate_corr"] = v.to_numpy()
            NITRATE_SOURCE[cast_id] = "CROSS-CHECK (L2/SUNA_calibrated; no step08 grid for this cast)"
            return out
    merged_path = L2_SUNA_ROOT / f"{cast_id}_SUNA_1s.csv"
    if merged_path.exists():
        v = _per_dbar(pd.read_csv(merged_path), "suna_nitrate")
        if v is not None and v.notna().any():
            out["suna_nitrate"] = v.to_numpy()
            NITRATE_SOURCE[cast_id] = "RAW merged (not calibrated)"
            return out
    NITRATE_SOURCE[cast_id] = "none"
    return out

frames: Dict[str, pd.DataFrame] = {}
cast_files = sorted(L2_1M_DOWN.glob("*_1m_down.cnv"))
print(f"Found {len(cast_files)} CTD downcast profiles.")

for f in cast_files:
    cast_id = f.name[: -len("_1m_down.cnv")]
    try:
        df = parse_cnv(f)
        depth_col = pick_depth(df)
        if depth_col is None:
            print(f"  {cast_id}: no depth column, skipped")
            continue
        df = attach_nitrate(df, cast_id, depth_col)
        frames[cast_id] = df
        print(f"  {cast_id}: {len(df)} bins, depth={depth_col}, nitrate: {NITRATE_SOURCE.get(cast_id, 'none')}")
    except Exception as exc:
        print(f"  {cast_id}: FAILED -> {exc!r}")


## 5. Per-cast profile plots + lab-friendly CSV

In [ ]:
def lab_friendly(df: pd.DataFrame) -> pd.DataFrame:
    keep = [c for c in df.columns if c in LAB_RENAME]
    out = df[keep].rename(columns=LAB_RENAME)
    return out


made: List[Path] = []
for cast_id, df in frames.items():
    depth_col = pick_depth(df)
    # CTD variables
    for candidates, label, unit in CTD_VARIABLES:
        xcol = pick_var(df, candidates)
        if xcol is not None:
            made += plot_single(df, xcol, depth_col, label, unit, cast_id)
    # nitrate where present
    ncol = pick_var(df, NITRATE_VARIABLE[0])
    if ncol is not None and df[ncol].notna().any():
        made += plot_single(df, ncol, depth_col, NITRATE_VARIABLE[1], NITRATE_VARIABLE[2], cast_id)
    # lab-friendly CSV
    lab = lab_friendly(df)
    if not lab.empty:
        lab_path = TABLE_ROOT / f"{cast_id}_readable.csv"
        lab.to_csv(lab_path, index=False, encoding="utf-8-sig")

print(f"Wrote {len(made)} per-cast figure files to {PLOT_ROOT}")


## 6. Multi-cast overlays (one figure per variable)

In [ ]:
overlay_made: List[Path] = []
for candidates, label, unit in CTD_VARIABLES + [NITRATE_VARIABLE]:
    overlay_made += plot_overlay(frames, candidates, label, unit)

print(f"Wrote {len(overlay_made)} overlay figures.")
print("\nFigures in        :", PLOT_ROOT)
print("Readable tables in:", TABLE_ROOT, f"({len(list(TABLE_ROOT.glob('*_readable.csv')))} files)")
for p in sorted(PLOT_ROOT.glob("_overlay__*")):
    print("  ", p.name)


## 7. Interactive profile explorer

In [ ]:
# ------------------------------------------------------------------ interactive profile explorer
try:
    import ipywidgets as W
    from IPython.display import Image as _Img, display as _disp
    _HAVE_W = not __import__("os").environ.get("SUNCAT_HEADLESS")   # no widgets in a headless run (tools/run_notebook.py)
except Exception:
    _HAVE_W=False; print("ipywidgets not installed; pip install ipywidgets then Restart + Run All.")

VARMAP={l:(cands,u) for cands,l,u in CTD_VARIABLES}
VARMAP[NITRATE_VARIABLE[1]]=(NITRATE_VARIABLE[0],NITRATE_VARIABLE[2])

def _profile_png(cast_id, var_label):
    import io
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    df=frames[cast_id]; dcol=pick_depth(df); cands,unit=VARMAP[var_label]; xcol=pick_var(df,cands)
    fig=Figure(figsize=(4.3,6.0)); FigureCanvasAgg(fig); ax=fig.subplots()
    if xcol is not None:
        d=df[[xcol,dcol]].apply(pd.to_numeric,errors="coerce").dropna().sort_values(dcol)
        if not d.empty: ax.plot(d[xcol],d[dcol],"-",lw=1.4,color="#1f77b4")
    else:
        ax.text(0.5,0.5,"not in this cast",ha="center",transform=ax.transAxes)
    ax.invert_yaxis(); ax.set_xlabel(f"{var_label} ({unit})"); ax.set_ylabel("Depth (m)")
    ax.xaxis.set_label_position("top"); ax.xaxis.tick_top(); ax.grid(alpha=GRID_A)
    ax.set_title(cast_id,fontsize=10)
    buf=io.BytesIO(); fig.savefig(buf,format="png",dpi=120,bbox_inches="tight"); buf.seek(0); return buf.getvalue()

if _HAVE_W and frames:
    _casts=sorted(frames)
    wc=W.Dropdown(options=_casts,value=_casts[0],description="cast")
    wv=W.Dropdown(options=list(VARMAP),value=list(VARMAP)[0],description="variable")
    def _r(cast_id,var_label): _disp(_Img(data=_profile_png(cast_id,var_label)))
    out=W.interactive_output(_r,{"cast_id":wc,"var_label":wv}); _disp(W.VBox([W.HBox([wc,wv]),out]))